# To perform and visualize analyses online 

## Before the experiment: get everything ready 

### imports

In [201]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.utils import instantiate
from hydra.core.config_store import ConfigStore
from hydra import compose, initialize
from IPython.display import display
import sys
import os
from pathlib import Path


### SET THIS PATH FROM THE CURRENT WORKING DIRECTORY TO THE REPO DIRECTORY
relative_repo_path = "GitRepos/simulation_closed_loop"

In [3]:



# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
full_repo_path = os.path.join(cwd, relative_repo_path)

if not os.path.exists(full_repo_path):
    raise ValueError(f"The specified relative path to the repo does not exist: {full_repo_path}\
                     Check workding dir and relative repo dir path variable.")

print(f"Working directoty: {cwd}")
print(f"Full repo path: {full_repo_path}")

# append repo path 
sys.path.append(full_repo_path)



# Initialize Hydra with the relative path to the config directory
config_path = os.path.join(relative_repo_path,"model_in_the_loop/config")
print(f"Config path relative to cwd: {config_path}")
if not os.path.exists(os.path.join(cwd,config_path)):
    raise ValueError(f"The specified config path does not exist: {os.path.join(cwd,config_path)}\
                     Check workding dir and config dir path variable.")

# Initialize Hydra
with initialize(version_base="1.3", config_path=config_path):
    # Compose the configuration
    cfg = compose(config_name="config")

# Print the config to verify it loaded correctly
print("Configuration loaded successfully:")



Working directoty: /gpfs01/euler/User/ssuhai
Full repo path: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop
Config path relative to cwd: GitRepos/simulation_closed_loop/model_in_the_loop/config
Configuration loaded successfully:


In [4]:
from model_in_the_loop.core import (DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper,
                                                     )

from model_in_the_loop.utils.plotting import show_all_rois_plot
from model_in_the_loop.utils.file_management import copy_rec_files,create_directory_structure

from model_in_the_loop.utils.transform_to_avi_stimulus import create_single_mei_avis_and_metadata
from model_in_the_loop.utils.simple_logging import log

### Create processing components (connect them to DB)

In [5]:
# create preprocessor
os.environ["DJ_SUPPORT_FILEPATH_MANAGEMENT"] = "TRUE"

dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore
                data_subfolders=cfg.data_subfolders, # type: ignore


                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore
                )



In [9]:
# # # Load config and tables
dj_table_holder.setup()

schema_name: ageuler_ssuhai_closed_loop_testing


/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:195: UserWarning: Stimulus offset not set. Assuming 0 offset. This is incorrect for the standard dense noise stimulus.
  warnings.warn(
/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:203: UserWarning: Stimulus offset not set. Assuming 0 offset. This is incorrect for the standard dense noise stimulus.
  warnings.warn(
/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:112: UserWarning: Values for ['bardx', 'bardy', 'velumsec', 'tmovedurs'] in `stim_dict` for stimulus `movingbar` are None. This may cause problems downstream.
  warnings.warn(f'Values for {missing_info} in `stim_dict` for stimulus `{stim_name}` are None. '
/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:39: UserWarning: Number of triggers in trial_info=8 must match ntrigger_rep=1.
  warnings.warn(msg)


preprocessing params:
 [{'preprocess_id': 1, 'fs_resample': 60, 'stim_names': ['gChirp', 'lChirp', 'movingbar', 'densenoise']}, {'preprocess_id': 2, 'window_length': 60, 'poly_order': 3, 'non_negative': 1, 'subtract_baseline': 0, 'standardize': 1, 'stim_names': ['mouse_cam']}]
Saving classifier to /gpfs01/euler/User/ssuhai/datajoint/rgc_classifier/rgc_classifier.pkl
Done setting up!


In [ ]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)


quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)

random_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    cfg=cfg,
    seeds=[111,222]
)

## During the experimet

### Move files from server to the repo 

In [ ]:
create_directory_structure(base_directory= cfg.DJ.userinfo.data_dir,)


copy_rec_files(
    recording_files_dir=cfg.paths.recording_files_dir,  # type: ignore
    destination_base=cfg.DJ.userinfo.data_dir,  # type: ignore
    full_dummy_ini_dir= os.path.join(cfg.paths.repo_directory, cfg.paths.dummy_ini_dir),  # type: ignore
)

### Preprocessing

In [ ]:
preprocessor.upload_iteration_metadata()

In [ ]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
if len(missing_keys) == 1:
    field_key = missing_keys[0]
    print(f"Missing field key found: {field_key}")
elif len(missing_keys) > 1:
    raise ValueError(f"Multiple missing fields found: {missing_keys}")
else:
    print("No missing fields found, using the last field key.")
    all_field_key = dj_table_holder("Field")().proj().fetch(as_dict=True)
    field_key = all_field_key[-1]
    print(f"Field key: {field_key}")

### Visualize cell tpe and quality

In [ ]:
from model_in_the_loop.core.gui import ExtendedAutoRoiGui
gui = ExtendedAutoRoiGui(
    dj_table_holder=dj_table_holder, 
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper],
    take_roi_rgba_from_this_analysis=quality_type_analysis_wrapper.name,
    field_key=field_key,
    canvas_width=30)

In [ ]:
display(gui.start_gui())

# The stimulus

### select roi ids and seds

In [ ]:
roi_seed = []
print(len(roi_seed))

In [ ]:
roi_id2mei_id,roi_id2mei_info = random_seed_mei_wrapper.select_subset_of_meis_for_each_roi()

In [ ]:
create_single_mei_avis_and_metadata(
    rois_seed=roi_seed,
    roi_id2mei_ids = roi_id2mei_id,    
    mei_data_container=random_seed_mei_wrapper.mei_data_container,
    stimulus_table=dj_table_holder("Stimulus")(),
    fit_gauss_2d_rf_table= dj_table_holder("FitGauss2DRF")(),
    abs_save_dir=cfg.paths.stimulus_output_dir,
)

# Save data

In [ ]:
# save data 
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = os.path.join(cfg.paths.repo_directory,"data/pipeline_saved_data_during_experiment",timestamp)
random_seed_mei_wrapper.save_all_data_to_dir(save_dir=save_dir)

# Clean up

In [ ]:
userinput = input("Cleanup? (y/n): ")
if userinput.lower() == 'y':
    dj_table_holder.clear_tables("all")
